# Libraries

In [ ]:
# Reinstall PyTorch stack with CUDA 12.8 support
!pip uninstall -y torch torchvision torchaudio torchao
!pip install -q \
    torch==2.10.0+cu128 \
    torchvision==0.25.0+cu128 \
    torchaudio==2.10.0+cu128 \
    torchao==0.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

# vLLM
!pip install -q vllm==0.18.0

# Dependencies
!pip install -q "protobuf>=5.26.1,<6.0.0" --break-system-packages

In [ ]:
# Standard library imports
import os
import gc
import json
import time
import random
import shutil
import subprocess

# Third-party libraries
import numpy as np
from vllm import LLM, SamplingParams
from huggingface_hub import login
from transformers import AutoTokenizer, set_seed
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Deep learning framework
import torch

# Global Variables

In [ ]:
# User & Repository Configuration
USER_NAME = "Abdelrahmanemam01"
EMAIL     = "abdulrahmanemam01@gmail.com"
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Model Selection
MODEL = "Llama-3.2-3B-Instruct"

# Model & Tokenizer Configuration
MODEL_ID     = "meta-llama/Llama-3.2-3B-Instruct"
TOKENIZER_ID = "meta-llama/Llama-3.2-3B-Instruct"

MODEL_NAME           = "Llama-3.2-3B-Instruct"
MODEL_NAME_IN_REPO   = "Llama-3.2-3B-Instruct-Original"
#COMPRESSION_METHOD   = "WANDA"
#MODEL_TYPE           = "Pruning"
#SPARSITY = 70
#PATH = f"Models/{SPARSITY}"

# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]

# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42

# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [ ]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [ ]:
set_reproducibility(SEED)

# Huggingface Login

In [ ]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

# GitHub login

In [ ]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

# Download Model

In [ ]:
"""local_path = snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=f"{PATH}/*",
    local_dir="/kaggle/working/model"
)"""

# Prompt Preprocessing

**DummyPrompt Tokenization**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

**RealPrompt Tokenization**

In [ ]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [ ]:
llm = LLM(
    model=MODEL_ID,
    tokenizer=TOKENIZER_ID,
    dtype="float16",
    max_model_len=256,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.85,
    attention_backend = "TRITON_ATTN",
    enable_prefix_caching = False
)
print("vLLM Initialized Successfully!")


sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

ttft_sampling_params = SamplingParams(
    temperature=0.0,
    seed = SEED,
    max_tokens=1,
    ignore_eos=True # Ensures it doesn't accidentally stop early
)

# Inference

In [ ]:
N_RUNS = 10
ttft_times, latency_times = [], []

for _ in range(N_RUNS):
    gc.collect()
    torch.cuda.empty_cache()
    
    # Warm-up
    _ = llm.generate(
        prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
        sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
        use_tqdm=False
    )

    # TTFT
    ttft_start = time.perf_counter()
    llm.generate(prompts=[{"prompt_token_ids": prompt_token_ids}], sampling_params=ttft_sampling_params, use_tqdm=False)
    ttft_times.append(time.perf_counter() - ttft_start)

    # Latency
    latency_start = time.perf_counter()
    llm.generate(prompts=[{"prompt_token_ids": prompt_token_ids}], sampling_params=sampling_params, use_tqdm=False)
    latency_times.append(time.perf_counter() - latency_start)

# Stats
ttft_arr, latency_arr = np.array(ttft_times), np.array(latency_times)

ttft_mean,       ttft_std       = ttft_arr.mean(),       ttft_arr.std()
latency_mean,    latency_std    = latency_arr.mean(),    latency_arr.std()

throughput_arr                  = (MAX_GENERATION_TOKENS - 1) / (latency_arr - ttft_arr)
throughput_mean, throughput_std = throughput_arr.mean(), throughput_arr.std()

input_tokens          = len(prompt_token_ids)
total_generated_tokens = MAX_GENERATION_TOKENS

In [ ]:
print(latency_times)

In [ ]:
print(ttft_times)

# Results

**Writing in json file**

In [ ]:
throughput_mean

In [ ]:
benchmark_results = {
    "model_name"            : MODEL_NAME,
    #"model_type"            : MODEL_TYPE,
    #"compression_method"    : COMPRESSION_METHOD,
    #"sparsity"              : SPARSITY,
    "input_token_count"     : input_tokens,
    "generated_token_count" : total_generated_tokens,
    "ttft_sec"              : f"{ttft_mean:.2f} ± {ttft_std:.2f}",
    "latency_sec"           : f"{latency_mean:.2f} ± {latency_std:.2f}",
    "throughput_tokens_per_sec" : f"{throughput_mean:.2f} ± {throughput_std:.2f}",
}


# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

**Push Results to GitHub Repository**

In [ ]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/CloudEvaluation/{MODEL}/{MODEL_NAME_IN_REPO}/Sparsity/{SPARSITY}"
source_file = OUTPUT_FILE 
remote_url = f"https://{token}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Added the cloud inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")